Degree:

file_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

Company:

file_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"




startup_success_rate = startup_count / (startup_count + closed_business_count)

startup_fail_rate = closed_business_count / (startup_count + closed_business_count)

job_success_rate_proxy = job_count / (job_count + job_lost_count)

job_fail_rate_proxy = job_lost_count / (job_count + job_lost_count)

# Fail 1

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ============================================================
# FILE PATHS
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 2008
END_YEAR = 2012

CHUNKSIZE = 100_000

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def clean_number(x):
    return pd.to_numeric(x, errors="coerce")


def percent(numerator, denominator):
    if pd.isna(numerator) or pd.isna(denominator) or denominator == 0:
        return np.nan
    return numerator / denominator * 100


def format_percent(x):
    if pd.isna(x):
        return ""
    return f"{x:.2f}%"


# ============================================================
# FIELD OF STUDY → INDUSTRY CONNECTION
# This is the bridge between Degree table and Company table.
# You can add/change mappings later.
# ============================================================

FIELD_TO_INDUSTRY_RULES = [
    {
        "field_keywords": ["computer", "information technology", "information sciences", "data"],
        "industry_keywords": ["information", "technology", "professional"],
        "connected_name": "Technology"
    },
    {
        "field_keywords": ["business", "management", "marketing", "finance", "accounting"],
        "industry_keywords": ["finance", "insurance", "management", "professional"],
        "connected_name": "Finance"
    },
    {
        "field_keywords": ["engineering", "mechanic", "construction"],
        "industry_keywords": ["manufacturing", "construction", "engineering"],
        "connected_name": "Manufacturing"
    },
    {
        "field_keywords": ["health", "nursing", "medicine", "medical"],
        "industry_keywords": ["health care", "healthcare", "social assistance"],
        "connected_name": "Healthcare"
    },
    {
        "field_keywords": ["education", "teacher"],
        "industry_keywords": ["educational services", "education"],
        "connected_name": "Education Services"
    },
    {
        "field_keywords": ["social sciences", "public administration", "legal", "law"],
        "industry_keywords": ["public administration", "government"],
        "connected_name": "Public Administration"
    },
    {
        "field_keywords": ["agriculture"],
        "industry_keywords": ["agriculture"],
        "connected_name": "Agriculture"
    },
    {
        "field_keywords": ["visual", "performing arts", "arts"],
        "industry_keywords": ["arts", "entertainment", "recreation"],
        "connected_name": "Arts / Entertainment"
    },
    {
        "field_keywords": ["communication", "journalism"],
        "industry_keywords": ["information"],
        "connected_name": "Information"
    },
    {
        "field_keywords": ["science", "biology", "physical sciences", "mathematics"],
        "industry_keywords": ["professional", "scientific", "technical"],
        "connected_name": "Professional / Scientific"
    },
]


def connect_field_to_industry(field_name):
    field_lower = clean_text(field_name).lower()

    for rule in FIELD_TO_INDUSTRY_RULES:
        for keyword in rule["field_keywords"]:
            if keyword in field_lower:
                return rule["connected_name"]

    return "Other / No direct industry match"


# ============================================================
# READ DEGREE FILE
# 2008-2012 only
# Uses real columns:
# year, field_group, major_name, degree_count, CTOTALT
# ============================================================

def read_degree_data(path):
    print("Loading Degree file...")

    degree_rows = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk.columns = chunk.columns.str.strip()

        if "year" not in chunk.columns:
            raise ValueError("Degree file does not have column: year")

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)]

        if chunk.empty:
            continue

        # Pick field column
        if "field_group" in chunk.columns:
            field_col = "field_group"
        elif "major_name" in chunk.columns:
            field_col = "major_name"
        else:
            raise ValueError("Degree file needs field_group or major_name column")

        # Pick count column
        # Prefer degree_count, but if missing or 0, use CTOTALT
        if "degree_count" in chunk.columns:
            chunk["degree_count_real"] = clean_number(chunk["degree_count"])
        else:
            chunk["degree_count_real"] = np.nan

        if "CTOTALT" in chunk.columns:
            chunk["ctotalt_real"] = clean_number(chunk["CTOTALT"])
        else:
            chunk["ctotalt_real"] = np.nan

        chunk["field_count"] = chunk["degree_count_real"]

        # If degree_count is blank or 0, use CTOTALT
        chunk.loc[
            chunk["field_count"].isna() | (chunk["field_count"] == 0),
            "field_count"
        ] = chunk["ctotalt_real"]

        temp = chunk[["year", field_col, "field_count"]].copy()
        temp = temp.rename(columns={field_col: "field_of_study_name"})

        temp["field_of_study_name"] = temp["field_of_study_name"].apply(clean_text)
        temp = temp[temp["field_of_study_name"] != ""]
        temp = temp[temp["field_count"].notna()]
        temp = temp[temp["field_count"] > 0]

        degree_rows.append(temp)

    if not degree_rows:
        raise ValueError("No Degree data found for 2008-2012")

    degree_df = pd.concat(degree_rows, ignore_index=True)

    degree_summary = (
        degree_df
        .groupby(["year", "field_of_study_name"], as_index=False)["field_count"]
        .sum()
    )

    degree_summary["connected_industry"] = degree_summary["field_of_study_name"].apply(
        connect_field_to_industry
    )

    print("Degree file loaded.")
    return degree_summary


# ============================================================
# COMPANY METRIC DETECTION
# These keywords decide which real Company rows become:
# startup_count, closed_business_count, job_count, job_lost_count
# ============================================================

STARTUP_KEYWORDS = [
    "startup", "start-up", "business birth", "firm birth",
    "establishment birth", "births", "formation", "openings", "entries"
]

CLOSED_KEYWORDS = [
    "closed", "closure", "closing", "business death", "firm death",
    "establishment death", "deaths", "exit", "exits", "shutdown"
]

JOB_POSITIVE_KEYWORDS = [
    "job creation", "job created", "jobs created",
    "gross job gains", "job gains", "employment gain",
    "employment increase", "openings"
]

JOB_NEGATIVE_KEYWORDS = [
    "job destruction", "job destroyed", "jobs lost",
    "job losses", "gross job losses", "employment loss",
    "employment decrease", "separations"
]

BAD_RATE_WORDS = [
    "rate", "percent", "percentage", "share", "ratio", "index"
]


def contains_any(text, keywords):
    text = clean_text(text).lower()
    return any(k in text for k in keywords)


def classify_company_metric(metric_name, metric_category=""):
    text = (clean_text(metric_name) + " " + clean_text(metric_category)).lower()

    # Do not use rate/percent rows as count rows
    if contains_any(text, BAD_RATE_WORDS):
        return None

    if contains_any(text, STARTUP_KEYWORDS):
        return "startup_count"

    if contains_any(text, CLOSED_KEYWORDS):
        return "closed_business_count"

    if contains_any(text, JOB_POSITIVE_KEYWORDS):
        return "job_count"

    if contains_any(text, JOB_NEGATIVE_KEYWORDS):
        return "job_lost_count"

    return None


def simplify_company_industry(industry_name):
    name = clean_text(industry_name).lower()

    if "information" in name or "technology" in name:
        return "Technology"

    if "finance" in name or "insurance" in name:
        return "Finance"

    if "manufacturing" in name:
        return "Manufacturing"

    if "health care" in name or "healthcare" in name or "social assistance" in name:
        return "Healthcare"

    if "education" in name or "educational services" in name:
        return "Education Services"

    if "public administration" in name or "government" in name:
        return "Public Administration"

    if "agriculture" in name:
        return "Agriculture"

    if "arts" in name or "entertainment" in name or "recreation" in name:
        return "Arts / Entertainment"

    if "professional" in name or "scientific" in name or "technical" in name:
        return "Professional / Scientific"

    return clean_text(industry_name)


# ============================================================
# READ COMPANY FILE
# 2008-2012 only
# Uses real columns:
# year, industry_name, metric_name, metric_category, value
# ============================================================

def read_company_data(path):
    print("Loading Company file...")

    company_rows = []
    metric_check_rows = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk.columns = chunk.columns.str.strip()

        required = ["year", "value", "metric_name"]
        for col in required:
            if col not in chunk.columns:
                raise ValueError(f"Company file missing required column: {col}")

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)]

        if chunk.empty:
            continue

        if "industry_name" not in chunk.columns:
            chunk["industry_name"] = "All Industries"

        if "metric_category" not in chunk.columns:
            chunk["metric_category"] = ""

        chunk["value"] = clean_number(chunk["value"])

        chunk["metric_type"] = chunk.apply(
            lambda r: classify_company_metric(
                r.get("metric_name", ""),
                r.get("metric_category", "")
            ),
            axis=1
        )

        used = chunk[chunk["metric_type"].notna()].copy()

        if used.empty:
            continue

        used["connected_industry"] = used["industry_name"].apply(simplify_company_industry)

        metric_check_rows.append(
            used[[
                "year",
                "industry_name",
                "connected_industry",
                "metric_name",
                "metric_category",
                "metric_type",
                "value"
            ]].head(100)
        )

        temp = used[[
            "year",
            "connected_industry",
            "metric_type",
            "value"
        ]].copy()

        company_rows.append(temp)

    if not company_rows:
        raise ValueError(
            "No Company metrics matched startup/closed/job keywords for 2008-2012. "
            "Print metric_name values and adjust keyword lists."
        )

    company_df = pd.concat(company_rows, ignore_index=True)

    company_summary = (
        company_df
        .groupby(["year", "connected_industry", "metric_type"], as_index=False)["value"]
        .sum()
    )

    company_wide = (
        company_summary
        .pivot_table(
            index=["year", "connected_industry"],
            columns="metric_type",
            values="value",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for col in [
        "startup_count",
        "closed_business_count",
        "job_count",
        "job_lost_count"
    ]:
        if col not in company_wide.columns:
            company_wide[col] = 0

    metric_check_df = pd.concat(metric_check_rows, ignore_index=True) if metric_check_rows else pd.DataFrame()

    print("Company file loaded.")
    return company_wide, metric_check_df


# ============================================================
# BUILD FINAL REAL TABLE USING FAKE TABLE FORMAT
# ============================================================

degree_summary = read_degree_data(DEGREE_FILE)
company_summary, metric_check_df = read_company_data(COMPANY_FILE)

final_table = degree_summary.merge(
    company_summary,
    on=["year", "connected_industry"],
    how="left"
)

# Fill missing company values with 0
for col in [
    "startup_count",
    "closed_business_count",
    "job_count",
    "job_lost_count"
]:
    final_table[col] = final_table[col].fillna(0)

# Startup success/fail rate
final_table["startup_total"] = (
    final_table["startup_count"] + final_table["closed_business_count"]
)

final_table["startup_success_rate_num"] = final_table.apply(
    lambda r: percent(r["startup_count"], r["startup_total"]),
    axis=1
)

final_table["startup_fail_rate_num"] = final_table.apply(
    lambda r: percent(r["closed_business_count"], r["startup_total"]),
    axis=1
)

# Job success/fail proxy
# This is a proxy because jobs do not exactly equal business success.
# Formula:
# job_success_rate_proxy = job_count / (job_count + job_lost_count)
# job_fail_rate_proxy    = job_lost_count / (job_count + job_lost_count)

final_table["job_total_proxy"] = (
    final_table["job_count"] + final_table["job_lost_count"]
)

final_table["job_success_rate_proxy_num"] = final_table.apply(
    lambda r: percent(r["job_count"], r["job_total_proxy"]),
    axis=1
)

final_table["job_fail_rate_proxy_num"] = final_table.apply(
    lambda r: percent(r["job_lost_count"], r["job_total_proxy"]),
    axis=1
)

# Format percentages
final_table["startup_success_rate"] = final_table["startup_success_rate_num"].apply(format_percent)
final_table["startup_fail_rate"] = final_table["startup_fail_rate_num"].apply(format_percent)
final_table["job_success_rate_proxy"] = final_table["job_success_rate_proxy_num"].apply(format_percent)
final_table["job_fail_rate_proxy"] = final_table["job_fail_rate_proxy_num"].apply(format_percent)

# Keep fake-table style columns
final_table = final_table[[
    "year",
    "field_of_study_name",
    "field_count",
    "connected_industry",
    "startup_count",
    "closed_business_count",
    "job_count",
    "startup_success_rate",
    "startup_fail_rate",
    "job_success_rate_proxy",
    "job_fail_rate_proxy"
]]

# Sort
final_table = final_table.sort_values(
    ["year", "connected_industry", "field_count"],
    ascending=[True, True, False]
).reset_index(drop=True)

# ============================================================
# PRINT RESULTS
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 120)

print("\nREAL FAKE-STYLE TABLE: DEGREE + COMPANY, 2008-2012")
display(final_table)

print("\nCHECK: REAL COMPANY METRICS USED")
display(metric_check_df.drop_duplicates().head(100))

# Optional save
output_file = Path.home() / "Downloads/real_fake_style_degree_company_2008_2012.csv"
final_table.to_csv(output_file, index=False)

print("\nSaved to:")
print(output_file)

# Fail 2

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ============================================================
# FILE PATHS
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 2008
END_YEAR = 2012
CHUNKSIZE = 100_000

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 150)


# ============================================================
# BASIC HELPERS
# ============================================================

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def clean_number(x):
    return pd.to_numeric(x, errors="coerce")


def safe_rate(part, total):
    if pd.isna(part) or pd.isna(total) or total == 0:
        return np.nan
    return part / total * 100


def fmt_percent(x):
    if pd.isna(x):
        return ""
    return f"{x:.2f}%"


# ============================================================
# DEGREE FIELD → INDUSTRY CONNECTION
# ============================================================

def connect_field_to_industry(field_name):
    name = clean_text(field_name).lower()

    if any(x in name for x in ["computer", "information sciences", "data"]):
        return "Technology"

    if any(x in name for x in ["business", "management", "marketing", "finance", "accounting"]):
        return "Finance"

    if any(x in name for x in ["engineering", "mechanic", "construction"]):
        return "Manufacturing"

    if any(x in name for x in ["health", "nursing", "medicine", "medical"]):
        return "Healthcare"

    if "education" in name:
        return "Education Services"

    if any(x in name for x in ["social sciences", "public administration", "legal", "law", "homeland security"]):
        return "Public Administration"

    if "agriculture" in name:
        return "Agriculture"

    if any(x in name for x in ["arts", "visual", "performing", "liberal arts", "humanities"]):
        return "Arts / Entertainment"

    if any(x in name for x in ["communication", "journalism"]):
        return "Information"

    if any(x in name for x in ["science", "biology", "mathematics", "physical sciences"]):
        return "Professional / Scientific"

    return "Other / No direct industry match"


# ============================================================
# COMPANY INDUSTRY CLEANER
# ============================================================

def simplify_company_industry(industry_name):
    name = clean_text(industry_name).lower()

    if "information" in name or "technology" in name:
        return "Technology"

    if "finance" in name or "insurance" in name:
        return "Finance"

    if "manufacturing" in name:
        return "Manufacturing"

    if "health care" in name or "healthcare" in name or "social assistance" in name:
        return "Healthcare"

    if "education" in name or "educational services" in name:
        return "Education Services"

    if "public administration" in name or "government" in name:
        return "Public Administration"

    if "agriculture" in name:
        return "Agriculture"

    if "arts" in name or "entertainment" in name or "recreation" in name:
        return "Arts / Entertainment"

    if "professional" in name or "scientific" in name or "technical" in name:
        return "Professional / Scientific"

    return clean_text(industry_name)


# ============================================================
# COMPANY METRIC CLASSIFIER
# This is the important fixed part.
# It checks metric_code, metric_name, metric_category, notes, source_table.
# ============================================================

BAD_RATE_WORDS = [
    "rate", "percent", "percentage", "share", "ratio", "index"
]

STARTUP_PATTERNS = [
    r"\bstartup\b",
    r"\bstart-up\b",
    r"\bbusiness birth",
    r"\bfirm birth",
    r"\bestablishment birth",
    r"\bbirths\b",
    r"\bformation",
    r"\bentry\b",
    r"\bentries\b",
    r"\bopenings\b",
    r"\bopen business",
    r"\bnew business",
    r"\bnew firm",
    r"\bnew establishment",
]

CLOSED_PATTERNS = [
    r"\bclosed\b",
    r"\bclosure\b",
    r"\bclosing\b",
    r"\bbusiness death",
    r"\bfirm death",
    r"\bestablishment death",
    r"\bdeaths\b",
    r"\bexit\b",
    r"\bexits\b",
    r"\bshutdown\b",
    r"\bshut down\b",
    r"\bclosed business",
    r"\bbusiness closing",
]

JOB_GAIN_PATTERNS = [
    r"\bjob creation\b",
    r"\bjobs created\b",
    r"\bjob created\b",
    r"\bgross job gains\b",
    r"\bjob gains\b",
    r"\bemployment gain\b",
    r"\bemployment increase\b",
    r"\bnet job creation\b",
]

JOB_LOSS_PATTERNS = [
    r"\bjob destruction\b",
    r"\bjobs destroyed\b",
    r"\bjob destroyed\b",
    r"\bgross job losses\b",
    r"\bjob losses\b",
    r"\bjobs lost\b",
    r"\bemployment loss\b",
    r"\bemployment decrease\b",
    r"\bseparations\b",
]


def has_pattern(text, patterns):
    return any(re.search(pattern, text) for pattern in patterns)


def classify_company_metric(row):
    text_cols = [
        "metric_code",
        "metric_name",
        "metric_category",
        "dataset",
        "source_table",
        "notes",
        "unit"
    ]

    text = " ".join(clean_text(row.get(col, "")) for col in text_cols).lower()

    # Do not use rate or percent rows as count rows
    if any(word in text for word in BAD_RATE_WORDS):
        return None

    # Closed first, because some rows may contain words like business + exits
    if has_pattern(text, CLOSED_PATTERNS):
        return "closed_business_count"

    if has_pattern(text, STARTUP_PATTERNS):
        return "startup_count"

    if has_pattern(text, JOB_LOSS_PATTERNS):
        return "job_loss_count"

    if has_pattern(text, JOB_GAIN_PATTERNS):
        return "job_count"

    return None


# ============================================================
# READ DEGREE FILE
# ============================================================

def read_degree_data(path):
    print("Loading Degree file...")

    all_rows = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk.columns = chunk.columns.str.strip()

        if "year" not in chunk.columns:
            raise ValueError("Degree file missing year column.")

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)]

        if chunk.empty:
            continue

        if "field_group" in chunk.columns:
            field_col = "field_group"
        elif "major_name" in chunk.columns:
            field_col = "major_name"
        else:
            raise ValueError("Degree file needs field_group or major_name.")

        if "degree_count" in chunk.columns:
            chunk["degree_count_real"] = clean_number(chunk["degree_count"])
        else:
            chunk["degree_count_real"] = np.nan

        if "CTOTALT" in chunk.columns:
            chunk["ctotalt_real"] = clean_number(chunk["CTOTALT"])
        else:
            chunk["ctotalt_real"] = np.nan

        # Real formula for field_count
        chunk["field_count"] = chunk["degree_count_real"]

        # If degree_count is blank or 0, use CTOTALT
        chunk.loc[
            chunk["field_count"].isna() | (chunk["field_count"] == 0),
            "field_count"
        ] = chunk["ctotalt_real"]

        temp = chunk[["year", field_col, "field_count"]].copy()
        temp = temp.rename(columns={field_col: "field_of_study_name"})

        temp["field_of_study_name"] = temp["field_of_study_name"].apply(clean_text)
        temp = temp[temp["field_of_study_name"] != ""]
        temp = temp[temp["field_count"].notna()]
        temp = temp[temp["field_count"] > 0]

        all_rows.append(temp)

    if not all_rows:
        raise ValueError("No Degree data found for 2008-2012.")

    degree_df = pd.concat(all_rows, ignore_index=True)

    degree_summary = (
        degree_df
        .groupby(["year", "field_of_study_name"], as_index=False)["field_count"]
        .sum()
    )

    degree_summary["connected_industry"] = degree_summary["field_of_study_name"].apply(
        connect_field_to_industry
    )

    print("Degree file loaded.")
    return degree_summary


# ============================================================
# READ COMPANY FILE
# ============================================================

def read_company_data(path):
    print("Loading Company file...")

    all_rows = []
    metric_check_rows = []
    all_metric_names = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk.columns = chunk.columns.str.strip()

        required_cols = ["year", "value", "metric_name"]
        for col in required_cols:
            if col not in chunk.columns:
                raise ValueError(f"Company file missing required column: {col}")

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)]

        if chunk.empty:
            continue

        if "industry_name" not in chunk.columns:
            chunk["industry_name"] = "All Industries"

        if "metric_category" not in chunk.columns:
            chunk["metric_category"] = ""

        if "metric_code" not in chunk.columns:
            chunk["metric_code"] = ""

        if "dataset" not in chunk.columns:
            chunk["dataset"] = ""

        if "source_table" not in chunk.columns:
            chunk["source_table"] = ""

        if "notes" not in chunk.columns:
            chunk["notes"] = ""

        if "unit" not in chunk.columns:
            chunk["unit"] = ""

        chunk["value"] = clean_number(chunk["value"])

        # Save all metric names for checking
        all_metric_names.append(
            chunk[[
                "metric_code",
                "metric_name",
                "metric_category",
                "unit"
            ]].drop_duplicates()
        )

        chunk["metric_type"] = chunk.apply(classify_company_metric, axis=1)

        used = chunk[chunk["metric_type"].notna()].copy()

        if used.empty:
            continue

        used["connected_industry"] = used["industry_name"].apply(simplify_company_industry)

        metric_check_rows.append(
            used[[
                "year",
                "industry_name",
                "connected_industry",
                "metric_code",
                "metric_name",
                "metric_category",
                "unit",
                "metric_type",
                "value"
            ]]
        )

        temp = used[[
            "year",
            "connected_industry",
            "metric_type",
            "value"
        ]].copy()

        all_rows.append(temp)

    all_metric_names_df = (
        pd.concat(all_metric_names, ignore_index=True)
        .drop_duplicates()
        .sort_values(["metric_name", "metric_code"])
        .reset_index(drop=True)
    )

    if not all_rows:
        print("\nNo company rows matched yet.")
        print("Look at this metric list and add the correct keywords:")
        display(all_metric_names_df.head(200))
        raise ValueError("No Company metrics matched startup/closed/job keywords.")

    company_df = pd.concat(all_rows, ignore_index=True)

    company_summary = (
        company_df
        .groupby(["year", "connected_industry", "metric_type"], as_index=False)["value"]
        .sum()
    )

    company_wide = (
        company_summary
        .pivot_table(
            index=["year", "connected_industry"],
            columns="metric_type",
            values="value",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for col in [
        "startup_count",
        "closed_business_count",
        "job_count",
        "job_loss_count"
    ]:
        if col not in company_wide.columns:
            company_wide[col] = 0

    metric_check_df = (
        pd.concat(metric_check_rows, ignore_index=True)
        .drop_duplicates()
        .sort_values(["year", "connected_industry", "metric_type"])
        .reset_index(drop=True)
    )

    print("Company file loaded.")
    return company_wide, metric_check_df, all_metric_names_df


# ============================================================
# BUILD FINAL TABLE
# ============================================================

degree_summary = read_degree_data(DEGREE_FILE)
company_summary, metric_check_df, all_metric_names_df = read_company_data(COMPANY_FILE)

final_table = degree_summary.merge(
    company_summary,
    on=["year", "connected_industry"],
    how="left"
)

for col in [
    "startup_count",
    "closed_business_count",
    "job_count",
    "job_loss_count"
]:
    final_table[col] = final_table[col].fillna(0)

# ============================================================
# REAL FORMULAS
# ============================================================

# Startup formula
final_table["startup_total_for_rate"] = (
    final_table["startup_count"] + final_table["closed_business_count"]
)

final_table["startup_success_rate_num"] = final_table.apply(
    lambda r: safe_rate(r["startup_count"], r["startup_total_for_rate"]),
    axis=1
)

final_table["startup_fail_rate_num"] = final_table.apply(
    lambda r: safe_rate(r["closed_business_count"], r["startup_total_for_rate"]),
    axis=1
)

# Job proxy formula
final_table["job_total_for_rate"] = (
    final_table["job_count"] + final_table["job_loss_count"]
)

final_table["job_success_rate_proxy_num"] = final_table.apply(
    lambda r: safe_rate(r["job_count"], r["job_total_for_rate"]),
    axis=1
)

final_table["job_fail_rate_proxy_num"] = final_table.apply(
    lambda r: safe_rate(r["job_loss_count"], r["job_total_for_rate"]),
    axis=1
)

# Format percentage columns
final_table["startup_success_rate"] = final_table["startup_success_rate_num"].apply(fmt_percent)
final_table["startup_fail_rate"] = final_table["startup_fail_rate_num"].apply(fmt_percent)
final_table["job_success_rate_proxy"] = final_table["job_success_rate_proxy_num"].apply(fmt_percent)
final_table["job_fail_rate_proxy"] = final_table["job_fail_rate_proxy_num"].apply(fmt_percent)


# ============================================================
# FINAL FAKE-STYLE TABLE
# Includes formula-check columns for now
# ============================================================

final_table = final_table[[
    "year",
    "field_of_study_name",
    "field_count",
    "connected_industry",

    "startup_count",
    "closed_business_count",
    "startup_total_for_rate",
    "startup_success_rate",
    "startup_fail_rate",

    "job_count",
    "job_loss_count",
    "job_total_for_rate",
    "job_success_rate_proxy",
    "job_fail_rate_proxy"
]]

final_table = final_table.sort_values(
    ["year", "connected_industry", "field_count"],
    ascending=[True, True, False]
).reset_index(drop=True)


# ============================================================
# WARNINGS
# ============================================================

startup_problem = final_table[
    (final_table["startup_count"] > 0) &
    (final_table["closed_business_count"] == 0)
]

job_problem = final_table[
    (final_table["job_count"] > 0) &
    (final_table["job_loss_count"] == 0)
]

print("\nREAL FAKE-STYLE TABLE WITH FORMULA CHECK COLUMNS")
display(final_table)

print("\nCHECK: COMPANY METRICS USED")
display(metric_check_df.head(200))

print("\nCHECK: ALL COMPANY METRIC NAMES FROM 2008-2012")
display(all_metric_names_df.head(300))

if len(startup_problem) > 0:
    print("\nWARNING:")
    print("Some rows have startup_count > 0 but closed_business_count = 0.")
    print("That means the Company file did not match the closed/death/exit metric yet.")
    print("Look at 'ALL COMPANY METRIC NAMES' and add the correct closed keyword.")

if len(job_problem) > 0:
    print("\nWARNING:")
    print("Some rows have job_count > 0 but job_loss_count = 0.")
    print("That means the Company file did not match the job loss/destruction metric yet.")
    print("Look at 'ALL COMPANY METRIC NAMES' and add the correct job-loss keyword.")


# ============================================================
# SAVE RESULT
# ============================================================

output_file = Path.home() / "Downloads/real_fake_style_degree_company_2008_2012_FIXED_FORMULA.csv"
final_table.to_csv(output_file, index=False)

print("\nSaved to:")
print(output_file)

# Fail 3

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    display = print

# ============================================================
# FILE PATHS
# ============================================================
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

YEARS = list(range(2008, 2013))
CHUNKSIZE = 200_000

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 300)


# ============================================================
# CHECK FILES
# ============================================================
def check_file(path, label):
    if not path.exists():
        raise FileNotFoundError(f"Missing {label} file:\n{path}")
    print(f"FOUND {label}: {path}")


check_file(DEGREE_FILE, "DEGREE")
check_file(COMPANY_FILE, "COMPANY")


# ============================================================
# CLEAN TEXT
# ============================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def norm_text(x):
    return clean_text(x).lower()


# ============================================================
# FIELD OF STUDY -> CONNECTED INDUSTRY
# You can edit this later, but this is only for checking table shape.
# ============================================================
def map_field_to_industry(field):
    f = norm_text(field)

    if "agriculture" in f:
        return "Agriculture"

    if "visual" in f or "performing arts" in f or "arts" in f or "humanities" in f:
        return "Arts / Entertainment"

    if "education" in f:
        return "Education Services"

    if "business" in f or "management" in f or "marketing" in f or "finance" in f:
        return "Finance"

    if "computer" in f or "information sciences" in f or "engineering" in f:
        return "Technology"

    if "health" in f or "biological" in f or "biomedical" in f:
        return "Health Care"

    if "construction" in f or "architecture" in f:
        return "Construction"

    if (
        "public administration" in f
        or "social service" in f
        or "legal" in f
        or "law" in f
        or "homeland" in f
        or "security" in f
        or "criminal justice" in f
        or "social sciences" in f
    ):
        return "Public Administration"

    return "Other"


# ============================================================
# COMPANY INDUSTRY -> CONNECTED INDUSTRY
# ============================================================
def map_company_industry(industry):
    x = norm_text(industry)

    if "agriculture" in x:
        return "Agriculture"

    if "arts" in x or "entertainment" in x or "recreation" in x:
        return "Arts / Entertainment"

    if "education" in x:
        return "Education Services"

    if "financial" in x or "finance" in x or "insurance" in x:
        return "Finance"

    if "information" in x or "technology" in x:
        return "Technology"

    if "health" in x or "social assistance" in x:
        return "Health Care"

    if "construction" in x:
        return "Construction"

    if "public administration" in x:
        return "Public Administration"

    return clean_text(industry)


# ============================================================
# CORRECT METRIC TYPE FORMULA
# This is the important fix.
# ============================================================
def classify_metric(metric_name):
    m = norm_text(metric_name)

    if m == "establishment births":
        return "startup_count"

    if m == "establishment deaths":
        return "closed_business_count"

    if m == "gross job gains":
        return "job_count"

    if m == "gross job losses":
        return "job_loss_count"

    # Do NOT use Openings as job_count.
    # Openings and Closings are establishment flow columns.
    if m == "openings":
        return "business_openings_check_only"

    if m == "closings":
        return "business_closings_check_only"

    return "other_check_only"


# ============================================================
# LOAD DEGREE PRE-FORMULA TABLE
# ============================================================
print("\nLoading DEGREE file...")

degree_header = pd.read_csv(DEGREE_FILE, nrows=0)
degree_cols = degree_header.columns.tolist()

print("\nDEGREE COLUMNS:")
display(pd.DataFrame({
    "column_number": range(1, len(degree_cols) + 1),
    "column_name": degree_cols
}))

# Pick best column names from your file
year_col = "year"

if "field_of_study_name" in degree_cols:
    field_col = "field_of_study_name"
elif "major_name" in degree_cols:
    field_col = "major_name"
elif "field_group" in degree_cols:
    field_col = "field_group"
elif "cipdesc" in degree_cols:
    field_col = "cipdesc"
else:
    raise ValueError("Cannot find field-of-study column. Check DEGREE columns above.")

if "field_count" in degree_cols:
    count_col = "field_count"
elif "degree_count" in degree_cols:
    count_col = "degree_count"
else:
    raise ValueError("Cannot find field_count or degree_count column. Check DEGREE columns above.")

degree_parts = []

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=[year_col, field_col, count_col],
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
    chunk[count_col] = pd.to_numeric(chunk[count_col], errors="coerce")

    chunk = chunk[
        chunk[year_col].isin(YEARS)
        & chunk[field_col].notna()
        & chunk[count_col].notna()
    ].copy()

    chunk = chunk.rename(columns={
        year_col: "year",
        field_col: "field_of_study_name",
        count_col: "field_count"
    })

    degree_parts.append(chunk)

degree_raw = pd.concat(degree_parts, ignore_index=True)

degree_pre = (
    degree_raw
    .groupby(["year", "field_of_study_name"], as_index=False)["field_count"]
    .sum()
)

degree_pre["connected_industry"] = degree_pre["field_of_study_name"].apply(map_field_to_industry)

print("\nCHECK 1: DEGREE PRE-FORMULA TABLE")
display(degree_pre.head(100))


# ============================================================
# LOAD COMPANY PRE-FORMULA TABLE
# ============================================================
print("\nLoading COMPANY file...")

company_header = pd.read_csv(COMPANY_FILE, nrows=0)
company_cols = company_header.columns.tolist()

print("\nCOMPANY COLUMNS:")
display(pd.DataFrame({
    "column_number": range(1, len(company_cols) + 1),
    "column_name": company_cols
}))

needed_company_cols = [
    "year",
    "industry_name",
    "metric_code",
    "metric_name",
    "metric_category",
    "unit",
    "value"
]

missing = [c for c in needed_company_cols if c not in company_cols]
if missing:
    raise ValueError(f"Missing columns in company file: {missing}")

company_parts = []

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=needed_company_cols,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk[
        chunk["year"].isin(YEARS)
        & chunk["industry_name"].notna()
        & chunk["metric_name"].notna()
        & chunk["value"].notna()
    ].copy()

    company_parts.append(chunk)

company_raw = pd.concat(company_parts, ignore_index=True)

company_raw["connected_industry"] = company_raw["industry_name"].apply(map_company_industry)
company_raw["metric_type"] = company_raw["metric_name"].apply(classify_metric)

print("\nCHECK 2: RAW COMPANY METRICS USED BEFORE FORMULA")
display(
    company_raw[
        [
            "year",
            "industry_name",
            "connected_industry",
            "metric_code",
            "metric_name",
            "metric_category",
            "unit",
            "metric_type",
            "value"
        ]
    ]
    .sort_values(["year", "connected_industry", "metric_name"])
    .head(300)
)


# ============================================================
# CHECK ALL METRIC NAMES 2008-2012
# ============================================================
print("\nCHECK 3: ALL COMPANY METRIC NAMES FROM 2008-2012")
metric_check = (
    company_raw[
        ["metric_code", "metric_name", "metric_category", "unit", "metric_type"]
    ]
    .drop_duplicates()
    .sort_values(["metric_category", "metric_name", "metric_code"])
    .reset_index(drop=True)
)

display(metric_check)


# ============================================================
# COMPANY PRE-FORMULA PIVOT
# This shows count columns BEFORE success/fail rate.
# ============================================================
company_for_formula = company_raw[
    company_raw["metric_type"].isin([
        "startup_count",
        "closed_business_count",
        "job_count",
        "job_loss_count"
    ])
].copy()

company_pre = (
    company_for_formula
    .groupby(["year", "connected_industry", "metric_type"], as_index=False)["value"]
    .sum()
    .pivot_table(
        index=["year", "connected_industry"],
        columns="metric_type",
        values="value",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for col in ["startup_count", "closed_business_count", "job_count", "job_loss_count"]:
    if col not in company_pre.columns:
        company_pre[col] = 0

company_pre["startup_total_for_rate"] = (
    company_pre["startup_count"] + company_pre["closed_business_count"]
)

company_pre["job_total_for_rate"] = (
    company_pre["job_count"] + company_pre["job_loss_count"]
)

company_pre = company_pre[
    [
        "year",
        "connected_industry",
        "startup_count",
        "closed_business_count",
        "startup_total_for_rate",
        "job_count",
        "job_loss_count",
        "job_total_for_rate"
    ]
]

print("\nCHECK 4: COMPANY PRE-FORMULA TABLE")
display(company_pre.sort_values(["year", "connected_industry"]).reset_index(drop=True))


# ============================================================
# FINAL PRE-FORMULA TABLE
# No success rate formula yet.
# This lets you check the numbers before calculation.
# ============================================================
pre_formula_table = degree_pre.merge(
    company_pre,
    on=["year", "connected_industry"],
    how="left"
)

number_cols = [
    "startup_count",
    "closed_business_count",
    "startup_total_for_rate",
    "job_count",
    "job_loss_count",
    "job_total_for_rate"
]

pre_formula_table[number_cols] = pre_formula_table[number_cols].fillna(0)

pre_formula_table = pre_formula_table[
    [
        "year",
        "field_of_study_name",
        "field_count",
        "connected_industry",
        "startup_count",
        "closed_business_count",
        "startup_total_for_rate",
        "job_count",
        "job_loss_count",
        "job_total_for_rate"
    ]
]

print("\nCHECK 5: FINAL PRE-FORMULA TABLE, NO RATE YET")
display(
    pre_formula_table
    .sort_values(["year", "connected_industry", "field_of_study_name"])
    .reset_index(drop=True)
)

# Fail 4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CHART CHECK:
# Shows which degree rows matched company data
# and which rows became missing because connected_industry did not match.
# ============================================================

# IMPORTANT:
# Do NOT use fillna(0) before this chart.
# We want missing to stay missing.

chart_df = degree_pre.merge(
    company_pre,
    on=["year", "connected_industry"],
    how="left",
    indicator=True
)

chart_df["company_match_status"] = np.where(
    chart_df["_merge"] == "both",
    "Matched company data",
    "No company match"
)

# ============================================================
# TABLE 1: MATCH VS NO MATCH BY YEAR
# ============================================================
match_by_year = (
    chart_df
    .groupby(["year", "company_match_status"])
    .size()
    .reset_index(name="degree_rows")
)

print("\nTABLE 1: MATCH VS NO COMPANY MATCH BY YEAR")
display(match_by_year)


# ============================================================
# CHART 1: MATCH VS NO MATCH BY YEAR
# ============================================================
match_pivot = (
    match_by_year
    .pivot(index="year", columns="company_match_status", values="degree_rows")
    .fillna(0)
)

match_pivot.plot(kind="bar", figsize=(10, 5))

plt.title("Degree Rows: Matched Company Data vs No Company Match")
plt.xlabel("Year")
plt.ylabel("Number of Degree Rows")
plt.xticks(rotation=0)
plt.legend(title="Match Status")
plt.tight_layout()
plt.show()


# ============================================================
# TABLE 2: WHICH CONNECTED INDUSTRIES ARE MISSING?
# ============================================================
missing_industry = (
    chart_df[chart_df["company_match_status"] == "No company match"]
    .groupby(["year", "connected_industry"])
    .size()
    .reset_index(name="missing_degree_rows")
    .sort_values(["year", "missing_degree_rows"], ascending=[True, False])
)

print("\nTABLE 2: CONNECTED INDUSTRIES WITH NO COMPANY MATCH")
display(missing_industry)


# ============================================================
# CHART 2: TOP MISSING CONNECTED INDUSTRIES
# ============================================================
top_missing = (
    chart_df[chart_df["company_match_status"] == "No company match"]
    .groupby("connected_industry")
    .size()
    .reset_index(name="missing_degree_rows")
    .sort_values("missing_degree_rows", ascending=False)
)

print("\nTABLE 3: TOTAL MISSING BY CONNECTED INDUSTRY")
display(top_missing)

top_missing.plot(
    kind="barh",
    x="connected_industry",
    y="missing_degree_rows",
    figsize=(10, 6),
    legend=False
)

plt.title("Degree Industries With No Matching Company Data")
plt.xlabel("Number of Degree Rows Missing Company Match")
plt.ylabel("Connected Industry")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


# ============================================================
# TABLE 4: MATCHED COMPANY DATA ONLY
# ============================================================
matched_only = chart_df[chart_df["company_match_status"] == "Matched company data"].copy()

matched_summary = (
    matched_only
    .groupby(["year", "connected_industry"], as_index=False)
    .agg(
        degree_rows=("field_of_study_name", "count"),
        total_field_count=("field_count", "sum"),
        startup_count=("startup_count", "first"),
        closed_business_count=("closed_business_count", "first"),
        job_count=("job_count", "first"),
        job_loss_count=("job_loss_count", "first")
    )
    .sort_values(["year", "connected_industry"])
)

print("\nTABLE 4: MATCHED DATA ONLY")
display(matched_summary)


# ============================================================
# CHART 3: STARTUP VS CLOSED BUSINESS BY INDUSTRY
# Pick one year to check.
# ============================================================
YEAR_TO_SHOW = 2008

startup_chart = (
    matched_summary[matched_summary["year"] == YEAR_TO_SHOW]
    .sort_values("startup_count", ascending=False)
)

startup_chart.plot(
    kind="barh",
    x="connected_industry",
    y=["startup_count", "closed_business_count"],
    figsize=(11, 7)
)

plt.title(f"Startup Count vs Closed Business Count by Industry, {YEAR_TO_SHOW}")
plt.xlabel("Count")
plt.ylabel("Connected Industry")
plt.gca().invert_yaxis()
plt.legend(["Startup Count", "Closed Business Count"])
plt.tight_layout()
plt.show()


# ============================================================
# CHART 4: JOB GAINS VS JOB LOSSES BY INDUSTRY
# ============================================================
job_chart = (
    matched_summary[matched_summary["year"] == YEAR_TO_SHOW]
    .sort_values("job_count", ascending=False)
)

job_chart.plot(
    kind="barh",
    x="connected_industry",
    y=["job_count", "job_loss_count"],
    figsize=(11, 7)
)

plt.title(f"Job Count vs Job Loss Count by Industry, {YEAR_TO_SHOW}")
plt.xlabel("Count")
plt.ylabel("Connected Industry")
plt.gca().invert_yaxis()
plt.legend(["Job Count", "Job Loss Count"])
plt.tight_layout()
plt.show()


# Fail 5

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ============================================================
# MEMORY-OPTIMIZED PRE-FORMULA JOB TABLE + CHARTS
# DOES NOT SAVE FILES
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000

DEGREE_COLS = [
    "year",
    "degree_count",
    "field_group",
    "field_subgroup",
    "major_name"
]

COMPANY_COLS = [
    "year",
    "industry_name",
    "metric_name",
    "value"
]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 120)


# ============================================================
# CHECK FILES
# ============================================================

print("Checking files...")

if not DEGREE_FILE.exists():
    raise FileNotFoundError(f"Missing Degree file: {DEGREE_FILE}")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking files.")
print("Degree file:", DEGREE_FILE)
print("Company file:", COMPANY_FILE)


# ============================================================
# CONNECT DEGREE FIELD TO INDUSTRY
# ============================================================

def connect_industry_from_text(text):
    text = str(text).lower()

    if "agriculture" in text:
        return "Agriculture"
    elif "visual" in text or "performing arts" in text or "liberal arts" in text or "humanities" in text:
        return "Arts / Entertainment"
    elif "education" in text:
        return "Education Services"
    elif "business" in text or "management" in text or "marketing" in text:
        return "Finance"
    elif "computer" in text or "information" in text or "engineering" in text:
        return "Technology"
    elif "health" in text or "medicine" in text or "nursing" in text:
        return "Health Care"
    elif "law" in text or "legal" in text:
        return "Legal Services"
    elif "construction" in text:
        return "Construction"
    elif "communication" in text or "journalism" in text:
        return "Information / Media"
    elif "social science" in text or "psychology" in text:
        return "Professional Services"
    else:
        return "Other / Unmapped"


# ============================================================
# LOAD DEGREE FILE IN CHUNKS
# ============================================================

print("\nLoading Degree file in chunks...")

degree_parts = []
degree_chunk_count = 0

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=DEGREE_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    degree_chunk_count += 1
    print(f"Loading Degree chunk {degree_chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce")

    chunk = chunk.dropna(subset=["year"])
    chunk["year"] = chunk["year"].astype(int)

    part = (
        chunk
        .groupby(["year", "field_group", "field_subgroup", "major_name"], dropna=False)["degree_count"]
        .sum()
        .reset_index()
        .rename(columns={"degree_count": "field_count"})
    )

    degree_parts.append(part)

print("Finished loading Degree chunks.")
print("Combining Degree chunks...")

degree_summary = pd.concat(degree_parts, ignore_index=True)

degree_summary = (
    degree_summary
    .groupby(["year", "field_group", "field_subgroup", "major_name"], dropna=False)["field_count"]
    .sum()
    .reset_index()
)

degree_summary["field_text"] = (
    degree_summary["field_group"].astype(str) + " " +
    degree_summary["field_subgroup"].astype(str) + " " +
    degree_summary["major_name"].astype(str)
)

degree_summary["connected_industry"] = degree_summary["field_text"].apply(connect_industry_from_text)

degree_summary = degree_summary.drop(columns=["field_text"])

print("Finished Degree summary.")
print("Degree years found:")
print(sorted(degree_summary["year"].dropna().unique()))
print("Degree summary rows:", len(degree_summary))


# ============================================================
# LOAD COMPANY FILE IN CHUNKS
# ============================================================

print("\nLoading Company file in chunks...")

company_parts = []
company_chunk_count = 0
matched_job_metric_names = set()
matched_job_loss_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    company_chunk_count += 1
    print(f"Loading Company chunk {company_chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    metric = chunk["metric_name"].astype(str).str.lower()

    # Avoid accidentally using rate/percent metrics as counts
    not_rate_mask = ~metric.str.contains(
        "rate|percent|percentage|pct|share|ratio",
        na=False
    )

    # Job count / job gain / job opening type metrics
    job_count_mask = metric.str.contains(
        "job|employment|opening|openings|hire|hires|hiring|gain|gains|created|creation",
        na=False
    ) & not_rate_mask

    # Job loss / job destruction / decline type metrics
    job_loss_mask = metric.str.contains(
        "job destruction|destruction|loss|lost|layoff|layoffs|separation|separations|decline|decrease|closed|closure",
        na=False
    ) & not_rate_mask

    matched_job_metric_names.update(chunk.loc[job_count_mask & ~job_loss_mask, "metric_name"].dropna().unique())
    matched_job_loss_metric_names.update(chunk.loc[job_loss_mask, "metric_name"].dropna().unique())

    chunk["job_count"] = np.where(
        job_count_mask & ~job_loss_mask,
        chunk["value"],
        0
    )

    chunk["job_loss_count"] = np.where(
        job_loss_mask,
        chunk["value"],
        0
    )

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["job_count", "job_loss_count"]]
        .sum()
        .reset_index()
        .rename(columns={"industry_name": "connected_industry"})
    )

    company_parts.append(part)

print("Finished loading Company chunks.")
print("Combining Company chunks...")

job_summary = pd.concat(company_parts, ignore_index=True)

job_summary = (
    job_summary
    .groupby(["year", "connected_industry"], dropna=False)[["job_count", "job_loss_count"]]
    .sum()
    .reset_index()
)

job_summary["job_total_for_rate"] = (
    job_summary["job_count"] + job_summary["job_loss_count"]
)

job_summary["job_data_status"] = np.where(
    job_summary["job_total_for_rate"].eq(0),
    "0 or missing job total",
    "usable for formula"
)

print("Finished Company job summary.")
print("Company job years found:")
print(sorted(job_summary["year"].dropna().unique()))
print("Company job summary rows:", len(job_summary))


# ============================================================
# SHOW WHICH METRIC_NAME VALUES WERE USED
# ============================================================

print("\nMetric names used for job_count:")
print(sorted(matched_job_metric_names))

print("\nMetric names used for job_loss_count:")
print(sorted(matched_job_loss_metric_names))


# ============================================================
# MERGE DEGREE + JOB PRE-FORMULA TABLE
# Keeps ALL Degree years
# ============================================================

print("\nMerging Degree summary with Company job summary...")

pre_formula_job_table = pd.merge(
    degree_summary,
    job_summary,
    on=["year", "connected_industry"],
    how="left"
)

pre_formula_job_table["job_count"] = pre_formula_job_table["job_count"].fillna(0)
pre_formula_job_table["job_loss_count"] = pre_formula_job_table["job_loss_count"].fillna(0)
pre_formula_job_table["job_total_for_rate"] = pre_formula_job_table["job_total_for_rate"].fillna(0)

pre_formula_job_table["job_data_status"] = np.where(
    pre_formula_job_table["job_total_for_rate"].eq(0),
    "0 or missing job total",
    "usable for formula"
)

cols_to_print = [
    "year",
    "major_name",
    "field_group",
    "field_subgroup",
    "field_count",
    "connected_industry",
    "job_count",
    "job_loss_count",
    "job_total_for_rate",
    "job_data_status"
]

pre_formula_job_table = pre_formula_job_table[cols_to_print].sort_values(
    ["year", "connected_industry", "major_name"]
)

print("Finished merging.")
print("\nFINAL PRE-FORMULA JOB TABLE")
print("Years printed:")
print(sorted(pre_formula_job_table["year"].dropna().unique()))
print("Total rows printed:", len(pre_formula_job_table))

display(pre_formula_job_table)


# ============================================================
# CHART 1: JOB COUNT VS JOB LOSS BY YEAR
# ============================================================

print("\nLoading chart 1: Job count vs job loss by year...")

chart_year = (
    pre_formula_job_table
    .groupby("year", dropna=False)[["job_count", "job_loss_count"]]
    .sum()
    .reset_index()
    .sort_values("year")
)

plt.figure(figsize=(14, 6))
plt.plot(chart_year["year"], chart_year["job_count"], marker="o", label="Job count")
plt.plot(chart_year["year"], chart_year["job_loss_count"], marker="o", label="Job loss count")
plt.title("Pre-formula Job Count vs Job Loss Count by Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.show()

print("Finished chart 1.")


# ============================================================
# CHART 2: JOB TOTAL BY CONNECTED INDUSTRY
# ============================================================

print("\nLoading chart 2: Job total by connected industry...")

chart_industry = (
    pre_formula_job_table
    .groupby("connected_industry", dropna=False)["job_total_for_rate"]
    .sum()
    .reset_index()
    .sort_values("job_total_for_rate", ascending=True)
)

plt.figure(figsize=(12, 7))
plt.barh(chart_industry["connected_industry"], chart_industry["job_total_for_rate"])
plt.title("Pre-formula Job Total by Connected Industry")
plt.xlabel("Job total for rate")
plt.ylabel("Connected industry")
plt.grid(axis="x")
plt.show()

print("Finished chart 2.")


# ============================================================
# CHART 3: FIELD COUNT VS JOB TOTAL
# ============================================================

print("\nLoading chart 3: Field count vs job total...")

chart_scatter = pre_formula_job_table[
    (pre_formula_job_table["field_count"] > 0) &
    (pre_formula_job_table["job_total_for_rate"] > 0)
].copy()

plt.figure(figsize=(10, 6))
plt.scatter(chart_scatter["field_count"], chart_scatter["job_total_for_rate"], alpha=0.5)
plt.title("Field Count vs Job Total for Rate")
plt.xlabel("Field count")
plt.ylabel("Job total for rate")
plt.grid(True)
plt.show()

print("Finished chart 3.")


# ============================================================
# FINISH
# ============================================================

print("\nALL DONE.")
print("Pre-formula table created: pre_formula_job_table")
print("Charts created:")
print("1. Job count vs job loss by year")
print("2. Job total by connected industry")
print("3. Field count vs job total")

# Fail 6

In [ ]:
# ============================================================
# SAVE CHARTS ONLY
# ============================================================

OUTPUT_DIR = Path.home() / "Downloads" / "pre_formula_job_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nSaving charts only...")
print("Chart folder:", OUTPUT_DIR)


# ============================================================
# CHART 1: JOB COUNT VS JOB LOSS BY YEAR
# ============================================================

print("Saving chart 1...")

chart_year = (
    pre_formula_job_table
    .groupby("year", dropna=False)[["job_count", "job_loss_count"]]
    .sum()
    .reset_index()
    .sort_values("year")
)

plt.figure(figsize=(14, 6))
plt.plot(chart_year["year"], chart_year["job_count"], marker="o", label="Job count")
plt.plot(chart_year["year"], chart_year["job_loss_count"], marker="o", label="Job loss count")
plt.title("Pre-formula Job Count vs Job Loss Count by Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "chart_1_job_count_vs_job_loss_by_year.png", dpi=300)
plt.close()

print("Finished chart 1.")


# ============================================================
# CHART 2: JOB TOTAL BY CONNECTED INDUSTRY
# ============================================================

print("Saving chart 2...")

chart_industry = (
    pre_formula_job_table
    .groupby("connected_industry", dropna=False)["job_total_for_rate"]
    .sum()
    .reset_index()
    .sort_values("job_total_for_rate", ascending=True)
)

plt.figure(figsize=(12, 7))
plt.barh(chart_industry["connected_industry"], chart_industry["job_total_for_rate"])
plt.title("Pre-formula Job Total by Connected Industry")
plt.xlabel("Job total for rate")
plt.ylabel("Connected industry")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "chart_2_job_total_by_connected_industry.png", dpi=300)
plt.close()

print("Finished chart 2.")


# ============================================================
# CHART 3: FIELD COUNT VS JOB TOTAL
# ============================================================

print("Saving chart 3...")

chart_scatter = pre_formula_job_table[
    (pre_formula_job_table["field_count"] > 0) &
    (pre_formula_job_table["job_total_for_rate"] > 0)
].copy()

plt.figure(figsize=(10, 6))
plt.scatter(chart_scatter["field_count"], chart_scatter["job_total_for_rate"], alpha=0.5)
plt.title("Field Count vs Job Total for Rate")
plt.xlabel("Field count")
plt.ylabel("Job total for rate")
plt.grid(True)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "chart_3_field_count_vs_job_total.png", dpi=300)
plt.close()

print("\nALL DONE.")
print("Charts saved in:", OUTPUT_DIR)
print("Saved files:")
print("1. chart_1_job_count_vs_job_loss_by_year.png")
print("2. chart_2_job_total_by_connected_industry.png")
print("3. chart_3_field_count_vs_job_total.png")

# Fail 7

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# RAW COMPANY 2015 CHART BY ALL INDUSTRY
# Uses only:
# Company.year
# Company.industry_name
# Company.metric_name
# Company.value
# ============================================================

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads" / "pre_formula_job_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000
YEAR_TO_CHART = 2015

COMPANY_COLS = [
    "year",
    "industry_name",
    "metric_name",
    "value"
]

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD ONLY 2015 RAW COMPANY DATA
# ============================================================

print("\nLoading raw Company data for 2015...")

parts = []
chunk_count = 0
matched_job_metric_names = set()
matched_job_loss_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    # Keep only 2015
    chunk = chunk[chunk["year"] == YEAR_TO_CHART].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Do not use rate/percent/share rows as count rows
    not_rate_mask = ~metric.str.contains(
        "rate|percent|percentage|pct|share|ratio",
        na=False
    )

    # Job count / job gain / job opening type rows
    job_count_mask = metric.str.contains(
        "job|employment|opening|openings|hire|hires|hiring|gain|gains|created|creation",
        na=False
    ) & not_rate_mask

    # Job loss / job destruction / closing type rows
    job_loss_mask = metric.str.contains(
        "job destruction|destruction|loss|lost|layoff|layoffs|separation|separations|decline|decrease|closed|closure",
        na=False
    ) & not_rate_mask

    matched_job_metric_names.update(
        chunk.loc[job_count_mask & ~job_loss_mask, "metric_name"].dropna().unique()
    )

    matched_job_loss_metric_names.update(
        chunk.loc[job_loss_mask, "metric_name"].dropna().unique()
    )

    chunk["job_count"] = np.where(
        job_count_mask & ~job_loss_mask,
        chunk["value"],
        0
    )

    chunk["job_loss_count"] = np.where(
        job_loss_mask,
        chunk["value"],
        0
    )

    part = (
        chunk
        .groupby("industry_name", dropna=False)[["job_count", "job_loss_count"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading 2015 raw Company data.")


# ============================================================
# COMBINE 2015 INDUSTRY DATA
# ============================================================

if not parts:
    raise ValueError("No 2015 Company data found with these raw columns.")

chart_2015_industry = pd.concat(parts, ignore_index=True)

chart_2015_industry = (
    chart_2015_industry
    .groupby("industry_name", dropna=False)[["job_count", "job_loss_count"]]
    .sum()
    .reset_index()
)

chart_2015_industry["job_total"] = (
    chart_2015_industry["job_count"] + chart_2015_industry["job_loss_count"]
)

chart_2015_industry = chart_2015_industry[
    chart_2015_industry["job_total"] > 0
].sort_values("job_total", ascending=True)

print("\nMetric names used for job_count:")
print(sorted(matched_job_metric_names))

print("\nMetric names used for job_loss_count:")
print(sorted(matched_job_loss_metric_names))

print("\nFinal 2015 industry rows used for chart:")
print(chart_2015_industry)


# ============================================================
# CHART: 2015 JOB COUNT VS JOB LOSS BY ALL INDUSTRY
# ============================================================

print("\nSaving 2015 raw Company industry chart...")

plt.figure(figsize=(14, 9))

plt.barh(
    chart_2015_industry["industry_name"],
    chart_2015_industry["job_count"],
    label="Job count"
)

plt.barh(
    chart_2015_industry["industry_name"],
    chart_2015_industry["job_loss_count"],
    left=chart_2015_industry["job_count"],
    label="Job loss count"
)

plt.title("2015 Raw Company Job Count and Job Loss by Industry")
plt.xlabel("Raw value count")
plt.ylabel("Industry name")
plt.legend()
plt.grid(axis="x")
plt.tight_layout()

save_path = OUTPUT_DIR / "chart_2015_raw_company_job_count_vs_job_loss_by_industry.png"
plt.savefig(save_path, dpi=300)
plt.close()

print("Finished chart.")
print("Saved chart here:")
print(save_path)

# Fail 8

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# STACKED / DIVERGING CHART
# 2020-2025
# LEFT = JOB LOSS
# RIGHT = JOB GAIN
# RAW COMPANY COLUMNS ONLY
# ============================================================

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads" / "pre_formula_job_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000
START_YEAR = 2020
END_YEAR = 2025

COMPANY_COLS = [
    "year",
    "industry_name",
    "metric_name",
    "value"
]

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY DATA 2020-2025
# ============================================================

print("\nLoading raw Company data from 2020 to 2025...")

parts = []
chunk_count = 0

matched_gain_metric_names = set()
matched_loss_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    # Keep only 2020-2025
    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Do not use percent/rate/share rows as count rows
    not_rate_mask = ~metric.str.contains(
        "rate|percent|percentage|pct|share|ratio",
        na=False
    )

    # Gain / job created / hiring / opening type rows
    gain_mask = metric.str.contains(
        "job|employment|opening|openings|hire|hires|hiring|gain|gains|created|creation",
        na=False
    ) & not_rate_mask

    # Loss / job destruction / layoff / closure type rows
    loss_mask = metric.str.contains(
        "job destruction|destruction|loss|lost|layoff|layoffs|separation|separations|decline|decrease|closed|closure",
        na=False
    ) & not_rate_mask

    # Save which raw metric_name values were used
    matched_gain_metric_names.update(
        chunk.loc[gain_mask & ~loss_mask, "metric_name"].dropna().unique()
    )

    matched_loss_metric_names.update(
        chunk.loc[loss_mask, "metric_name"].dropna().unique()
    )

    # Create chart columns from raw value
    chunk["job_gain"] = np.where(
        gain_mask & ~loss_mask,
        chunk["value"],
        0
    )

    chunk["job_loss"] = np.where(
        loss_mask,
        chunk["value"],
        0
    )

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading raw Company data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError("No Company data found from 2020 to 2025.")

chart_data = pd.concat(parts, ignore_index=True)

chart_data = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

chart_data["job_total"] = chart_data["job_gain"] + chart_data["job_loss"]

chart_data = chart_data[chart_data["job_total"] > 0].copy()

print("\nMetric names used for RIGHT SIDE job_gain:")
print(sorted(matched_gain_metric_names))

print("\nMetric names used for LEFT SIDE job_loss:")
print(sorted(matched_loss_metric_names))

print("\nFinal rows used for chart:")
print(chart_data)


# ============================================================
# SAVE ONE CHART PER YEAR
# LEFT = LOSS
# RIGHT = GAIN
# ============================================================

for year in range(START_YEAR, END_YEAR + 1):

    year_data = chart_data[chart_data["year"] == year].copy()

    if year_data.empty:
        print(f"No data for {year}, skipping chart.")
        continue

    year_data = year_data.sort_values("job_total", ascending=True)

    plt.figure(figsize=(14, 9))

    # LEFT SIDE: loss as negative
    plt.barh(
        year_data["industry_name"],
        -year_data["job_loss"],
        label="Job loss"
    )

    # RIGHT SIDE: gain as positive
    plt.barh(
        year_data["industry_name"],
        year_data["job_gain"],
        label="Job gain"
    )

    plt.axvline(0, linewidth=1)
    plt.title(f"{year} Job Loss vs Job Gain by Industry")
    plt.xlabel("Count: loss on left, gain on right")
    plt.ylabel("Industry")
    plt.legend()
    plt.grid(axis="x")
    plt.tight_layout()

    save_path = OUTPUT_DIR / f"chart_{year}_left_loss_right_gain_by_industry.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved chart for {year}: {save_path}")


# ============================================================
# SAVE COMBINED 2020-2025 CHART BY YEAR
# LEFT = TOTAL LOSS
# RIGHT = TOTAL GAIN
# ============================================================

year_summary = (
    chart_data
    .groupby("year", dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
    .sort_values("year")
)

plt.figure(figsize=(12, 6))

plt.barh(
    year_summary["year"].astype(str),
    -year_summary["job_loss"],
    label="Job loss"
)

plt.barh(
    year_summary["year"].astype(str),
    year_summary["job_gain"],
    label="Job gain"
)

plt.axvline(0, linewidth=1)
plt.title("2020-2025 Job Loss vs Job Gain")
plt.xlabel("Count: loss on left, gain on right")
plt.ylabel("Year")
plt.legend()
plt.grid(axis="x")
plt.tight_layout()

combined_save_path = OUTPUT_DIR / "chart_2020_2025_left_loss_right_gain_by_year.png"
plt.savefig(combined_save_path, dpi=300)
plt.close()

print("\nALL DONE.")
print("Charts saved in:", OUTPUT_DIR)
print("Combined chart saved here:")
print(combined_save_path)

# success 1 - job gain job loss by industry

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DIVERGING STACKED BAR CHART
# Y = YEAR
# COLORS = INDUSTRY
# LEFT = JOB LOSS
# RIGHT = JOB GAIN
# JOB ONLY (NOT STARTUP / COMPANY CLOSE)
# ============================================================

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads" / "pre_formula_job_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000
START_YEAR = 2020
END_YEAR = 2025

COMPANY_COLS = [
    "year",
    "industry_name",
    "metric_name",
    "value"
]

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY DATA
# ============================================================

print(f"\nLoading raw Company job data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0

matched_job_gain_metric_names = set()
matched_job_loss_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate/percent/share rows
    not_rate_mask = ~metric.str.contains(
        "rate|percent|percentage|pct|share|ratio",
        na=False
    )

    # Exclude startup / business / company rows
    not_startup_company_mask = ~metric.str.contains(
        "startup|start-up|business|firm|company|establishment|birth|death|closed|closure|exit|entry|formation",
        na=False
    )

    # JOB GAIN ONLY
    job_gain_mask = metric.str.contains(
        "job gain|job gains|jobs gained|job creation|jobs created|employment gain|employment gains|hire|hires|hiring|job opening|job openings",
        na=False
    ) & not_rate_mask & not_startup_company_mask

    # JOB LOSS ONLY
    job_loss_mask = metric.str.contains(
        "job loss|job losses|jobs lost|job destruction|employment loss|employment losses|layoff|layoffs|separation|separations",
        na=False
    ) & not_rate_mask & not_startup_company_mask

    matched_job_gain_metric_names.update(
        chunk.loc[job_gain_mask, "metric_name"].dropna().unique()
    )

    matched_job_loss_metric_names.update(
        chunk.loc[job_loss_mask, "metric_name"].dropna().unique()
    )

    chunk["job_gain"] = np.where(job_gain_mask, chunk["value"], 0)
    chunk["job_loss"] = np.where(job_loss_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company job data found from {START_YEAR} to {END_YEAR}.")

chart_data = pd.concat(parts, ignore_index=True)

chart_data = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

chart_data["job_total"] = chart_data["job_gain"] + chart_data["job_loss"]
chart_data = chart_data[chart_data["job_total"] > 0].copy()

print("\nMetric names used for job_gain:")
print(sorted(matched_job_gain_metric_names))

print("\nMetric names used for job_loss:")
print(sorted(matched_job_loss_metric_names))


# ============================================================
# OPTIONAL: KEEP ALL INDUSTRIES
# If too crowded, set TOP_N = 10 or 15
# ============================================================

TOP_N = None   # Example: 12

if TOP_N is not None:
    top_industries = (
        chart_data.groupby("industry_name")["job_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    chart_data["industry_name"] = np.where(
        chart_data["industry_name"].isin(top_industries),
        chart_data["industry_name"],
        "Other"
    )

    chart_data = (
        chart_data
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

gain_pivot = (
    chart_data
    .pivot_table(index="year", columns="industry_name", values="job_gain", aggfunc="sum", fill_value=0)
    .sort_index()
)

loss_pivot = (
    chart_data
    .pivot_table(index="year", columns="industry_name", values="job_loss", aggfunc="sum", fill_value=0)
    .sort_index()
)

# Make sure both have the same industry columns
all_industries = sorted(set(gain_pivot.columns).union(set(loss_pivot.columns)))
gain_pivot = gain_pivot.reindex(columns=all_industries, fill_value=0)
loss_pivot = loss_pivot.reindex(columns=all_industries, fill_value=0)

print("\nYears used:")
print(gain_pivot.index.tolist())

print("\nIndustries used:")
print(all_industries)


# ============================================================
# PLOT DIVERGING STACKED HORIZONTAL BAR CHART
# ============================================================

plt.figure(figsize=(16, 9))

years = gain_pivot.index.astype(str)
y_pos = np.arange(len(years))

# One color per industry
colors = plt.cm.tab20(np.linspace(0, 1, len(all_industries)))

# RIGHT SIDE = job gain
right_base = np.zeros(len(gain_pivot))

for i, industry in enumerate(all_industries):
    values = gain_pivot[industry].values
    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )
    right_base += values

# LEFT SIDE = job loss
left_base = np.zeros(len(loss_pivot))

for i, industry in enumerate(all_industries):
    values = loss_pivot[industry].values
    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )
    left_base += values

plt.axvline(0, color="black", linewidth=1)
plt.yticks(y_pos, years)
plt.title(f"{START_YEAR}-{END_YEAR} Stacked Industry Job Loss vs Job Gain by Year")
plt.xlabel("Job loss on left   |   Job gain on right")
plt.ylabel("Year")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.legend(title="Industry", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

save_path = OUTPUT_DIR / f"chart_{START_YEAR}_{END_YEAR}_stacked_industry_left_loss_right_gain_by_year.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close()

print("\nALL DONE.")
print("Saved chart here:")
print(save_path)

# Success 2 - Startup & death Company

# Success 2 - Startup & death Company

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DIVERGING STACKED BAR CHART
# Y = YEAR
# COLORS = INDUSTRY
# LEFT = COMPANY DEATH / CLOSED BUSINESS
# RIGHT = STARTUP / NEW BUSINESS
# COMPANY STARTUP + DEATH ONLY
# NOT JOB GAIN / JOB LOSS
# ============================================================

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads" / "pre_formula_startup_death_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000
START_YEAR = 2020
END_YEAR = 2025

# ============================================================
# COLUMNS USED
# ============================================================
# Company.year          = year
# Company.industry_name = industry group / chart color
# Company.metric_name   = tells if row is startup or death
# Company.value         = number/count used in the chart

COMPANY_COLS = [
    "year",
    "industry_name",
    "metric_name",
    "value"
]

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY STARTUP / DEATH DATA
# ============================================================

print(f"\nLoading raw Company startup/death data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0

matched_startup_metric_names = set()
matched_death_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate / percent / share rows
    # We only want raw counts, not rate values
    not_rate_mask = ~metric.str.contains(
        "rate|percent|percentage|pct|share|ratio",
        na=False
    )

    # Exclude job rows
    # This chart is company/startup/death only, NOT jobs
    not_job_mask = ~metric.str.contains(
        "job|jobs|employment|hire|hires|hiring|layoff|layoffs|separation|separations",
        na=False
    )

    # STARTUP / NEW BUSINESS / NEW COMPANY
    startup_mask = metric.str.contains(
        "startup|start-up|birth|births|formation|formations|entry|entries|new business|new firm|new company|opening|openings",
        na=False
    ) & not_rate_mask & not_job_mask

    # COMPANY DEATH / CLOSED BUSINESS
    death_mask = metric.str.contains(
        "death|deaths|closed|closure|closures|closing|closings|exit|exits|failure|failures",
        na=False
    ) & not_rate_mask & not_job_mask

    matched_startup_metric_names.update(
        chunk.loc[startup_mask, "metric_name"].dropna().unique()
    )

    matched_death_metric_names.update(
        chunk.loc[death_mask, "metric_name"].dropna().unique()
    )

    chunk["startup_count"] = np.where(startup_mask, chunk["value"], 0)
    chunk["death_count"] = np.where(death_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company startup/death data found from {START_YEAR} to {END_YEAR}.")

chart_data = pd.concat(parts, ignore_index=True)

chart_data = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
)

chart_data["company_total"] = chart_data["startup_count"] + chart_data["death_count"]
chart_data = chart_data[chart_data["company_total"] > 0].copy()

print("\nMetric names used for startup_count:")
print(sorted(matched_startup_metric_names))

print("\nMetric names used for death_count:")
print(sorted(matched_death_metric_names))


# ============================================================
# OPTIONAL: KEEP ALL INDUSTRIES
# If too crowded, set TOP_N = 10 or 15
# ============================================================

TOP_N = None   # Example: 12

if TOP_N is not None:
    top_industries = (
        chart_data.groupby("industry_name")["company_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    chart_data["industry_name"] = np.where(
        chart_data["industry_name"].isin(top_industries),
        chart_data["industry_name"],
        "Other"
    )

    chart_data = (
        chart_data
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

startup_pivot = (
    chart_data
    .pivot_table(index="year", columns="industry_name", values="startup_count", aggfunc="sum", fill_value=0)
    .sort_index()
)

death_pivot = (
    chart_data
    .pivot_table(index="year", columns="industry_name", values="death_count", aggfunc="sum", fill_value=0)
    .sort_index()
)

# Make sure both have the same industry columns
all_industries = sorted(set(startup_pivot.columns).union(set(death_pivot.columns)))
startup_pivot = startup_pivot.reindex(columns=all_industries, fill_value=0)
death_pivot = death_pivot.reindex(columns=all_industries, fill_value=0)

print("\nYears used:")
print(startup_pivot.index.tolist())

print("\nIndustries used:")
print(all_industries)


# ============================================================
# PLOT DIVERGING STACKED HORIZONTAL BAR CHART
# ============================================================

plt.figure(figsize=(16, 9))

years = startup_pivot.index.astype(str)
y_pos = np.arange(len(years))

# One color per industry
colors = plt.cm.tab20(np.linspace(0, 1, len(all_industries)))

# RIGHT SIDE = startup / new business
right_base = np.zeros(len(startup_pivot))

for i, industry in enumerate(all_industries):
    values = startup_pivot[industry].values
    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )
    right_base += values

# LEFT SIDE = death / closed business
left_base = np.zeros(len(death_pivot))

for i, industry in enumerate(all_industries):
    values = death_pivot[industry].values
    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )
    left_base += values

plt.axvline(0, color="black", linewidth=1)
plt.yticks(y_pos, years)
plt.title(f"{START_YEAR}-{END_YEAR} Stacked Industry Company Death vs Startup by Year")
plt.xlabel("Company death / closed business on left   |   Startup / new business on right")
plt.ylabel("Year")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.legend(title="Industry", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

save_path = OUTPUT_DIR / f"chart_{START_YEAR}_{END_YEAR}_stacked_industry_left_death_right_startup_by_year.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close()

print("\nALL DONE.")
print("Saved chart here:")
print(save_path)


# ============================================================
# PRINT SMALL CHECK TABLE
# ============================================================

check_table = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
    .sort_values(["year", "industry_name"])
)

print("\nPre-formula check table:")
print(check_table.head(50))

# Successful 3 - Check how many unique instustry

In [ ]:
from pathlib import Path
import pandas as pd

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

COLS = ["year", "industry_code", "industry_name"]
CHUNKSIZE = 100_000

START_YEAR = 2020
END_YEAR = 2025

pairs = []

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk = chunk.dropna(subset=["year", "industry_code", "industry_name"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    pairs.append(chunk[["industry_code", "industry_name"]].drop_duplicates())

all_pairs = pd.concat(pairs, ignore_index=True).drop_duplicates()

print("Unique industry_name count:")
print(all_pairs["industry_name"].nunique())

print("\nUnique industry_code count:")
print(all_pairs["industry_code"].nunique())

print("\nIndustry code + industry name pairs:")
print(all_pairs.sort_values(["industry_name", "industry_code"]).to_string(index=False))

print("\nHow many industry_code values inside each industry_name:")
check = (
    all_pairs
    .groupby("industry_name")["industry_code"]
    .nunique()
    .reset_index(name="number_of_industry_codes")
    .sort_values("number_of_industry_codes", ascending=False)
)

print(check.to_string(index=False))

# FAil 9

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# DEGREE CHART IMAGES ONLY
# MAIN COLUMN = YEAR
# ONE CHART IMAGE FOR EACH OTHER COLUMN
# CHART TYPE = STACKED BAR
# VALUE = SUM OF degree_count
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_chart_images_only",

    "START_YEAR": 2020,
    "END_YEAR": 2025,   # 2025 may be empty if file stops at 2024

    "MAIN_COLUMN": "year",
    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    # keep top N categories in each chart so chart stays readable
    "TOP_N": 50,

    # chart size
    "FIG_WIDTH": 16,
    "FIG_HEIGHT": 9,
    "DPI": 200,

    # style
    "TITLE_SIZE": 16,
    "LABEL_SIZE": 12,
    "LEGEND_SIZE": 9,
    "ROTATION": 45,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

print("Loading Degree CSV...")

all_parts = []

for chunk_num, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Reading chunk {chunk_num}...")

    chunk.columns = chunk.columns.str.strip()

    keep_cols = [c for c in CFG["NEEDED_COLUMNS"] if c in chunk.columns]
    temp = chunk[keep_cols].copy()

    if "year" not in temp.columns or "degree_count" not in temp.columns:
        raise ValueError("Missing required columns: year or degree_count")

    temp["year"] = pd.to_numeric(temp["year"], errors="coerce")
    temp["degree_count"] = pd.to_numeric(temp["degree_count"], errors="coerce").fillna(0)

    temp = temp[
        (temp["year"] >= CFG["START_YEAR"]) &
        (temp["year"] <= CFG["END_YEAR"])
    ]

    if not temp.empty:
        all_parts.append(temp)

print("Finished loading.")

if not all_parts:
    print("No rows found for selected years.")
else:
    df = pd.concat(all_parts, ignore_index=True)

    for col in CFG["NEEDED_COLUMNS"]:
        if col not in df.columns:
            df[col] = ""

    df = df[CFG["NEEDED_COLUMNS"]].copy()

    print("Rows found:", len(df))
    print("Year range found:", int(df["year"].min()), "-", int(df["year"].max()))

    for group_col in CFG["OTHER_COLUMNS"]:
        print(f"Making chart for: {group_col}")

        temp = df[["year", group_col, "degree_count"]].copy()

        # clean category column
        temp[group_col] = temp[group_col].fillna("Missing").astype(str).str.strip()
        temp.loc[temp[group_col] == "", group_col] = "Missing"

        # total by category across all selected years
        top_categories = (
            temp.groupby(group_col, as_index=False)["degree_count"]
            .sum()
            .sort_values("degree_count", ascending=False)
            .head(CFG["TOP_N"])[group_col]
            .tolist()
        )

        temp[group_col] = temp[group_col].where(temp[group_col].isin(top_categories), "Other")

        chart_df = (
            temp.groupby(["year", group_col], as_index=False)["degree_count"]
            .sum()
            .pivot(index="year", columns=group_col, values="degree_count")
            .fillna(0)
            .sort_index()
        )

        # keep stable column order: top categories first, then Other if exists
        ordered_cols = [c for c in top_categories if c in chart_df.columns]
        if "Other" in chart_df.columns:
            ordered_cols.append("Other")
        chart_df = chart_df[ordered_cols]

        plt.figure(figsize=(CFG["FIG_WIDTH"], CFG["FIG_HEIGHT"]))
        chart_df.plot(
            kind="bar",
            stacked=True,
            figsize=(CFG["FIG_WIDTH"], CFG["FIG_HEIGHT"])
        )

        plt.title(
            f"Degree Count by Year and {group_col} (Top {CFG['TOP_N']})",
            fontsize=CFG["TITLE_SIZE"]
        )
        plt.xlabel("Year", fontsize=CFG["LABEL_SIZE"])
        plt.ylabel("Degree Count", fontsize=CFG["LABEL_SIZE"])
        plt.xticks(rotation=CFG["ROTATION"])
        plt.legend(
            title=group_col,
            fontsize=CFG["LEGEND_SIZE"],
            title_fontsize=CFG["LEGEND_SIZE"],
            bbox_to_anchor=(1.02, 1),
            loc="upper left"
        )
        plt.tight_layout()

        out_file = CFG["OUTPUT_DIR"] / f"{group_col}_stacked_bar_2020_2025.png"
        plt.savefig(out_file, dpi=CFG["DPI"], bbox_inches="tight")
        plt.close()

        print("Saved:", out_file)

    print("Done.")
    print("Output folder:", CFG["OUTPUT_DIR"])

# Success 4 - degree
| Code purpose        | Column name        | Meaning                                                  |
| ------------------- | ------------------ | -------------------------------------------------------- |
| Main year column    | `year`             | Groups data by year                                      |
| Number/value column | `degree_count`     | The degree amount/count being added                      |
| Category 1          | `major_name`       | Major name                                               |
| Category 2          | `field_group`      | Big field group                                          |
| Category 3          | `field_subgroup`   | Smaller field group                                      |
| Category 4          | `degree_group`     | Degree type/group                                        |
| Category 5          | `award_level_name` | Award level, like certificate, associate, bachelor, etc. |
| Category 6          | `cip_code`         | CIP code for the major/field                             |


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# DEGREE COUNT BY YEAR + ALL CATEGORIES
# - NO TOP N
# - NO OTHER BUCKET
# - REMOVE UNKNOWN / BLANK / NAN
# - PRINT TABLE
# - SAVE FULL CSV
# - SAVE IMAGE
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_count_all_table_and_image",

    "START_YEAR": 2020,
    "END_YEAR": 2025,   # 2025 will be 0 if file stops at 2024

    "MAIN_COLUMN": "year",
    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    # print
    "PRINT_ALL_ROWS": True,   # True = print full table
    "PRINT_MAX_ROWS": None,   # None = unlimited

    # chart
    "FIGSIZE": (20, 10),
    "DPI": 200,
    "SHOW_LEGEND": True,      # can be too large for big columns
    "LEGEND_FONT_SIZE": 8,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

# ============================================================
# PANDAS DISPLAY
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

if CFG["PRINT_ALL_ROWS"]:
    pd.set_option("display.max_rows", CFG["PRINT_MAX_ROWS"])
else:
    pd.set_option("display.max_rows", 100)

# ============================================================
# LOAD DATA
# ============================================================

print("Reading degree file...")

all_chunks = []

for i, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        usecols=CFG["NEEDED_COLUMNS"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Reading chunk {i}...")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk["year"] >= CFG["START_YEAR"]) &
        (chunk["year"] <= CFG["END_YEAR"])
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)
    all_chunks.append(chunk)

if not all_chunks:
    raise ValueError("No data found in selected year range.")

df = pd.concat(all_chunks, ignore_index=True)

print("Finished loading.")
print("Rows found:", len(df))
print("Year range found:", df["year"].min(), "-", df["year"].max())

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_col(series):
    """
    Convert to string, strip spaces, and standardize blanks/nan/none.
    """
    s = series.astype(str).str.strip()
    s = s.replace({
        "": "Unknown",
        "nan": "Unknown",
        "NaN": "Unknown",
        "None": "Unknown",
        "NULL": "Unknown",
        "null": "Unknown"
    })
    return s.fillna("Unknown")

def comma_format(x, pos):
    return f"{int(x):,}"

def make_table_and_image(col_name):
    print("\n" + "=" * 100)
    print(f"TABLE + IMAGE FOR: {col_name}")
    print("=" * 100)

    temp = df[["year", "degree_count", col_name]].copy()

    # clean text
    temp[col_name] = clean_col(temp[col_name])

    # remove Unknown / blank / nan
    temp = temp[
        ~temp[col_name].astype(str).str.strip().str.lower().isin(
            ["unknown", "nan", "none", ""]
        )
    ].copy()

    if temp.empty:
        print(f"No data left for {col_name} after removing Unknown/blank.")
        return

    # group
    grouped = (
        temp
        .groupby(["year", col_name], as_index=False)["degree_count"]
        .sum()
    )

    # pivot table for output
    table = grouped.pivot_table(
        index=col_name,
        columns="year",
        values="degree_count",
        aggfunc="sum",
        fill_value=0
    )

    years = list(range(CFG["START_YEAR"], CFG["END_YEAR"] + 1))

    for y in years:
        if y not in table.columns:
            table[y] = 0

    table = table[years]
    table[f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}"] = table[years].sum(axis=1)

    # sort biggest total first
    table = table.sort_values(
        by=f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}",
        ascending=False
    )

    table_reset = table.reset_index()

    # --------------------------------------------------------
    # PRINT TABLE
    # --------------------------------------------------------
    display(table_reset)

    # save csv
    csv_path = CFG["OUTPUT_DIR"] / f"{col_name}_degree_count_table_ALL.csv"
    table_reset.to_csv(csv_path, index=False)
    print("CSV saved:", csv_path)

    # --------------------------------------------------------
    # IMAGE
    # --------------------------------------------------------
    image_table = table.drop(columns=[f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}"])

    # transpose so x-axis = year
    plot_df = image_table.T

    fig, ax = plt.subplots(figsize=CFG["FIGSIZE"])

    plot_df.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.75
    )

    ax.set_title(f"Degree Count by Year and {col_name}", fontsize=20)
    ax.set_xlabel("Year", fontsize=14)
    ax.set_ylabel("Degree Count", fontsize=14)

    # remove scientific notation like 1e7
    ax.ticklabel_format(style="plain", axis="y", useOffset=False)
    ax.yaxis.set_major_formatter(FuncFormatter(comma_format))

    plt.xticks(rotation=45)

    # total on top of each year bar
    totals = plot_df.sum(axis=1)
    max_total = totals.max() if len(totals) > 0 else 0

    for x, total in enumerate(totals):
        if total > 0:
            ax.text(
                x,
                total + (max_total * 0.01 if max_total > 0 else 0),
                f"{int(total):,}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

    # legend
    if CFG["SHOW_LEGEND"]:
        ax.legend(
            title=col_name,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            fontsize=CFG["LEGEND_FONT_SIZE"]
        )
        plt.subplots_adjust(left=0.08, right=0.72, top=0.90, bottom=0.15)
    else:
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        plt.subplots_adjust(left=0.08, right=0.95, top=0.90, bottom=0.15)

    image_path = CFG["OUTPUT_DIR"] / f"{col_name}_degree_count_image_ALL.png"
    plt.savefig(image_path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.show()
    plt.close()

    print("Image saved:", image_path)

# ============================================================
# RUN
# ============================================================

for col in CFG["OTHER_COLUMNS"]:
    make_table_and_image(col)

print("\nDone.")
print("Saved in:", CFG["OUTPUT_DIR"])